# Đo đạc QC hệ thống VNU Đăng ký học phần trên Kaggle

Notebook dựng toàn bộ stack trong một máy Kaggle (PostgreSQL, Spring Boot, k6), chạy `load/k6/full_scenario.js`, đo tài nguyên, kiểm bất biến dữ liệu, chấm QC theo ngưỡng của kịch bản k6 và vẽ biểu đồ.

**Giới hạn:** backend, PostgreSQL và k6 chạy chung một máy Kaggle. Kết quả không phải số đo bốn máy qua Tailscale và không có độ trễ mạng Tailscale.

**Cách dùng:**
1. Bật `Settings > Internet > On` (cần xác minh số điện thoại tài khoản Kaggle).
2. Chọn `Run All` hoặc `Save Version > Save & Run All (Commit)`. Mã nguồn đã nhúng sẵn trong cell 2, không cần tải lên.
3. Kết quả nằm trong `/kaggle/working/load_results.zip` (tab Output). Giải nén vào `load/results/` của repo.


## Bước 1. Tham số chạy

Đặt số người dùng ảo (`VUS`), số tài khoản sinh viên (`STUDENTS`), phiên bản k6, thư mục làm việc và các mật khẩu ngẫu nhiên của lần chạy. Hàm `sh()` chạy lệnh shell và báo lỗi rõ.

| Tham số | Mặc định | Ý nghĩa |
|---|---|---|
| `VUS` | 1000 | Người dùng ảo đồng thời (đề bài) |
| `STUDENTS` | 1000 | Tài khoản `SV00001`..`SV01000` nạp vào DB |
| `WORK` | `/tmp/dkhp` | Thư mục tạm, không đưa vào Output |

*Đầu ra: thông tin máy (HĐH, số CPU).*

In [ ]:
# 1. Tham số. Sửa ở đây rồi Run all.
import os, sys, json, time, shutil, subprocess, secrets, pathlib, platform

VUS = 1000              # số người dùng ảo (đề bài: 1000)
STUDENTS = 1000         # số tài khoản sinh viên nạp vào DB
GIT_HASH = "b3438c5"    # commit gốc của mã nguồn nhúng ở cell 2 (kèm thay đổi chưa commit)
K6_VERSION = "0.54.0"

WORK = pathlib.Path(os.environ.get("DKHP_WORK", "/tmp/dkhp"))   # ngoài /kaggle/working để raw.csv không vào Output
OUT_DIR = pathlib.Path("/kaggle/working")
SRC = WORK / "src"
RESULTS = WORK / "results"
CHARTS = RESULTS / "charts"
for d in (SRC, CHARTS):
    d.mkdir(parents=True, exist_ok=True)

STUDENT_PASSWORD = "LoadTest2026"
OWNER_PASSWORD = secrets.token_hex(12)
API_DB_PASSWORD = secrets.token_hex(12)
JWT_SECRET = secrets.token_urlsafe(48)
BASE_URL = "http://127.0.0.1:8080/api/v1"

def sh(cmd, check=True, env=None, cwd=None, show=True):
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", text=True,
                       capture_output=True, env=env, cwd=cwd)
    if show:
        if r.stdout.strip(): print(r.stdout[-3000:])
        if r.stderr.strip(): print(r.stderr[-1500:])
    if check and r.returncode:
        raise RuntimeError(f"Lệnh lỗi ({r.returncode}): {cmd}\n{r.stderr[-1500:]}")
    return r

PG_ENV = {**os.environ, "PGHOST": "127.0.0.1", "PGPASSWORD": OWNER_PASSWORD}
print("Máy:", platform.platform(), "| CPU:", os.cpu_count())

## Bước 2. Nạp mã nguồn

Mã nguồn dự án (backend Spring Boot, SQL khởi tạo, dữ liệu mẫu, kịch bản k6) được nhúng sẵn dưới dạng zip base64 trong cell, nên không cần tải lên hay dataset. Cell giải nén vào `/tmp/dkhp/src` và kiểm tra các file bắt buộc.

*Đầu ra: `Mã nguồn sẵn sàng`. Thiếu file nào thì cell dừng và nêu tên file.*

In [ ]:
# 2. Nạp mã nguồn nhúng sẵn (zip base64, tạo từ commit b3438c5 cùng thay đổi chưa commit)
import base64, io, zipfile
PAYLOAD = "..."
if not (SRC / "pom.xml").exists():
    zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))).extractall(SRC)
sh(r"sed -i 's/\r$//' mvnw docker/initdb/tmp2_roles.sh", cwd=SRC)
for f in ("pom.xml", "docker/initdb/tmp1_init.sql", "docker/seed/seed_dev.sql",
          "docker/verify/invariants.sql", "load/k6/full_scenario.js", "load/k6/accounts.json"):
    assert (SRC / f).exists(), f"Thiếu {f} trong mã nguồn nhúng"
print("Mã nguồn sẵn sàng:", SRC)


### Bước 2b. Kiểm tra Internet

Kaggle mặc định tắt Internet. Các bước sau cần tải gói `apt`, Maven và k6. Nếu lỗi, bật `Settings > Internet > On` rồi chạy lại.

*Đầu ra: `Internet: OK`.*

In [ ]:
# 2b. Kaggle cần Internet bật để apt, Maven và k6 tải được
import urllib.request
try:
    urllib.request.urlopen("https://github.com", timeout=8)
except Exception as e:
    raise RuntimeError("Không có Internet. Bật Settings > Internet > On rồi chạy lại.") from e
print("Internet: OK")


## Bước 3. Cài công cụ

Cài JDK, PostgreSQL, k6 và thư viện phân tích (`psutil`, `pandas`, `matplotlib`).

*Đầu ra: phiên bản Java, k6, psql. Mất khoảng 40 giây.*

In [ ]:
# 3. Cài JDK 17, PostgreSQL, k6 và thư viện vẽ biểu đồ
sh("apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
   "openjdk-17-jdk-headless postgresql postgresql-contrib unzip curl > /dev/null")
if not shutil.which("k6"):
    url = f"https://github.com/grafana/k6/releases/download/v{K6_VERSION}/k6-v{K6_VERSION}-linux-amd64.tar.gz"
    sh(f"curl -sL {url} | tar xz -C /tmp && mv /tmp/k6-v{K6_VERSION}-linux-amd64/k6 /usr/local/bin/k6")
sh("pip install -q psutil matplotlib pandas")
print(sh("java -version 2>&1 | head -1; k6 version; psql --version", show=False).stdout)

## Bước 4. Dựng cơ sở dữ liệu

1. Khởi động PostgreSQL và tạo role chủ sở hữu `dkhp_app`.
2. Nạp schema `docker/initdb/tmp1_init.sql`.
3. Tạo role ứng dụng `dkhp_api` bằng `tmp2_roles.sh` (đúng phân quyền như triển khai thật).
4. Nạp dữ liệu mẫu `seed_dev.sql`: 1000 sinh viên, 8 môn, 12 lớp, đợt đăng ký `OPEN`.

*Đầu ra: bảng đếm `students | admins | courses | sections | period_status`.*

In [ ]:
# 4. Dựng PostgreSQL: schema (tmp1_init.sql), tài khoản API (tmp2_roles.sh), dữ liệu mẫu
sh("service postgresql start && sleep 3")
pathlib.Path("/tmp/role.sql").write_text(f"CREATE ROLE dkhp_app LOGIN SUPERUSER PASSWORD '{OWNER_PASSWORD}';\n")
os.chmod("/tmp/role.sql", 0o644)
sh("su postgres -c 'psql -f /tmp/role.sql' && su postgres -c 'createdb -O dkhp_app dkhp'")
sh("psql -U dkhp_app -d dkhp -v ON_ERROR_STOP=1 -q -f docker/initdb/tmp1_init.sql", env=PG_ENV, cwd=SRC)
role_env = {**PG_ENV, "POSTGRES_USER": "dkhp_app", "POSTGRES_DB": "dkhp",
            "DB_USER": "dkhp_api", "DB_PASSWORD": API_DB_PASSWORD}
sh("sh docker/initdb/tmp2_roles.sh", env=role_env, cwd=SRC)
sh(f"psql -U dkhp_app -d dkhp -v ON_ERROR_STOP=1 -q -v pw='{STUDENT_PASSWORD}' "
   f"-v students={STUDENTS} -v period_status=OPEN -f docker/seed/seed_dev.sql", env=PG_ENV, cwd=SRC)

## Bước 5. Build backend

Chạy `./mvnw package` (bỏ qua test) để tạo file jar Spring Boot.

*Đầu ra: tên và dung lượng jar. Mất khoảng 25 giây.*

In [ ]:
# 5. Build jar
sh("sh ./mvnw -B -ntp -q -DskipTests package", cwd=SRC)
JAR = next((SRC / "target").glob("course-registration-*.jar"))
print("Jar:", JAR.name, round(JAR.stat().st_size / 1e6, 1), "MB")

## Bước 6. Chạy backend

Khởi động jar với giới hạn tốc độ nâng rất cao. Mặc định 600 req/phút/IP và 10 lần đăng nhập/phút/tài khoản sẽ chặn gần hết tải k6 bằng HTTP 429, nên không đo được ứng dụng. Chờ `/health` trả 200 rồi đăng nhập thử `SV00001`.

*Đầu ra: `Backend sẵn sàng` và `Đăng nhập thử SV00001: 200`.*

In [ ]:
# 6. Chạy backend. Nâng giới hạn tốc độ để đo ứng dụng:
#    mặc định 600 req/phút/IP và 10 lần đăng nhập/phút/tài khoản sẽ chặn 429 gần hết tải của k6.
import urllib.request, urllib.error
app_env = {**os.environ,
    "DB_URL": "jdbc:postgresql://127.0.0.1:5432/dkhp", "DB_USER": "dkhp_api", "DB_PASSWORD": API_DB_PASSWORD,
    "JWT_SECRET": JWT_SECRET, "SERVER_ADDRESS": "127.0.0.1",
    "HEALTH_ALLOWED_IPS": "127.0.0.1,::1",
    "RATE_LIMIT_IP": "100000000", "RATE_LIMIT_ACCOUNT": "100000000", "RATE_LIMIT_LOGIN": "100000000"}
app_log = open(RESULTS / "app.log", "w")
app = subprocess.Popen(["java", "-Xmx2g", "-jar", str(JAR)], env=app_env, stdout=app_log, stderr=subprocess.STDOUT)
ok = False
for _ in range(90):
    try:
        if urllib.request.urlopen(BASE_URL + "/health", timeout=2).status == 200:
            ok = True; break
    except Exception:
        time.sleep(2)
assert ok, "Backend không lên được, xem " + str(RESULTS / "app.log")
print("Backend sẵn sàng, pid", app.pid)

def api(method, path, body=None, token=None):
    req = urllib.request.Request(BASE_URL + path, method=method,
        data=json.dumps(body).encode() if body is not None else None,
        headers={"Content-Type": "application/json", **({"Authorization": "Bearer " + token} if token else {})})
    try:
        r = urllib.request.urlopen(req, timeout=10); return r.status, r.read().decode()
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode()

status, body = api("POST", "/auth/login", {"code": "SV00001", "password": STUDENT_PASSWORD})
assert status == 200, (status, body)
print("Đăng nhập thử SV00001: 200")

## Bước 7. Chạy tải k6 và lấy mẫu tài nguyên

Chạy `load/k6/full_scenario.js`: 1000 VU, tăng 30 giây, giữ 60 giây, giảm 30 giây. Mỗi VU đăng nhập, xem đợt, xem lớp, đăng ký (có `Idempotency-Key`) rồi hủy. Song song, một luồng ghi CPU và RAM của Java, PostgreSQL và toàn máy mỗi giây, kèm số kết nối DB.

**Lưu ý:** k6, backend và PostgreSQL dùng chung 4 vCPU, nên độ trễ cao hơn so với chạy tách máy.
`k6 exit code 99` nghĩa là vi phạm ngưỡng của script nhưng vẫn có số liệu.

In [ ]:
# 7. Chạy k6 và lấy mẫu CPU/RAM mỗi giây
import threading, psutil
import pandas as pd

stop = threading.Event()
samples = []
def pg_conns():
    r = sh('psql -U dkhp_app -d dkhp -Atc "SELECT count(*) FROM pg_stat_activity WHERE datname=\'dkhp\'"',
           env=PG_ENV, show=False, check=False)
    try: return int(r.stdout.strip())
    except ValueError: return None

def sampler():
    jp = psutil.Process(app.pid)
    psutil.cpu_percent(None); jp.cpu_percent(None); n = 0
    while not stop.is_set():
        time.sleep(1); n += 1
        pgs = [p for p in psutil.process_iter(["name", "memory_info"]) if p.info["name"] == "postgres"]
        samples.append({"t": time.time(), "sys_cpu": psutil.cpu_percent(None),
               "java_cpu": jp.cpu_percent(None), "java_rss_mb": jp.memory_info().rss / 1e6,
               "pg_rss_mb": sum(p.info["memory_info"].rss for p in pgs if p.info["memory_info"]) / 1e6,
               "pg_conns": pg_conns() if n % 3 == 0 else None})
threading.Thread(target=sampler, daemon=True).start()

k6_cmd = (f"k6 run -e BASE_URL={BASE_URL} -e STUDENT_PASSWORD='{STUDENT_PASSWORD}' -e VUS={VUS} "
          f"--summary-export={RESULTS}/summary.json --out csv={WORK}/raw.csv load/k6/full_scenario.js")
t_start = time.time()
k6 = subprocess.run(k6_cmd, shell=True, executable="/bin/bash", cwd=SRC, text=True, capture_output=True)
t_end = time.time()
stop.set(); time.sleep(1.5)
(RESULTS / "k6_stdout.txt").write_text(k6.stdout + "\n" + k6.stderr)
print(k6.stdout[-3500:]); print(k6.stderr[-800:])
print("k6 exit code:", k6.returncode, "(99 = vi phạm ngưỡng p95/p99/error_rate của script, vẫn có số liệu)")
pd.DataFrame(samples).to_csv(RESULTS / "resources.csv", index=False)

## Bước 8. Đọc dữ liệu thô

Nạp `raw.csv` của k6, giữ chỉ số `http_req_duration` và `vus`, gộp đường dẫn thành endpoint (`/enrollments/123` thành `/enrollments/{id}`) và lưu `requests.csv.gz`.

*Đầu ra: bảng số yêu cầu theo endpoint.*

In [ ]:
# 8. Đọc kết quả thô của k6
import numpy as np
usecols = ["metric_name", "timestamp", "metric_value", "name", "status", "method"]
keep = {"http_req_duration", "vus"}
parts = []
for ch in pd.read_csv(WORK / "raw.csv", usecols=usecols, chunksize=500_000, low_memory=False):
    parts.append(ch[ch.metric_name.isin(keep)])
raw = pd.concat(parts, ignore_index=True)
raw["ts"] = pd.to_datetime(raw.timestamp, unit="s")
raw["sec"] = (raw.ts - raw.ts.min()).dt.total_seconds()

req = raw[raw.metric_name == "http_req_duration"].copy()
req["status"] = pd.to_numeric(req.status, errors="coerce").fillna(0).astype(int)
req["path"] = (req.name.astype(str).str.replace(r"^https?://[^/]+", "", regex=True)
                   .str.replace(r"\?.*$", "", regex=True).str.replace(r"/\d+$", "/{id}", regex=True)
                   .str.replace("/api/v1", "", regex=False))
req["endpoint"] = req.method.astype(str) + " " + req.path
req = req.rename(columns={"metric_value": "ms"})
req[["sec", "endpoint", "status", "ms"]].to_csv(RESULTS / "requests.csv.gz", index=False)
print(len(req), "request;", req.endpoint.nunique(), "endpoint")
req.groupby("endpoint").size()

## Bước 9. Chỉ số tổng hợp

Tính thông lượng (trung bình, đỉnh), độ trễ p50/p95/p99/max toàn hệ thống và từng endpoint, đếm mã HTTP, tỷ lệ lỗi (không tính 409 và 422 vì là kết quả nghiệp vụ hợp lệ), riêng bước đăng nhập, tài nguyên và thông tin môi trường. Ghi `stats.json`.

In [ ]:
# 9. Chỉ số tổng hợp
def pct(s): return {"p50": s.quantile(.5), "p95": s.quantile(.95), "p99": s.quantile(.99), "max": s.max()}
def r1(d): return {k: round(float(v), 1) for k, v in d.items()}
dur = req.sec.max() - req.sec.min()
per_sec = req.groupby(req.sec.astype(int)).size()
ok2xx = req.status.between(200, 299)
expected_biz = req.status.isin([409, 422])
login = req[req.endpoint == "POST /auth/login"]
stats = {
    "total_requests": int(len(req)), "duration_s": round(float(dur), 1),
    "throughput_avg_rps": round(len(req) / dur, 1), "throughput_peak_rps": int(per_sec.max()),
    "latency_ms": r1(pct(req.ms)),
    "success_2xx": int(ok2xx.sum()),
    "non_2xx_rate_pct": round(100 * (~ok2xx).mean(), 2),
    "error_rate_excl_business_pct": round(100 * (~ok2xx & ~expected_biz).mean(), 2),
    "status_counts": {str(k): int(v) for k, v in req.status.value_counts().sort_index().items()},
    "login_ms": r1(pct(login.ms)), "login_requests": int(len(login)),
    "login_non_200_pct": round(100 * (login.status != 200).mean(), 2),
    "enroll_requests": int((req.endpoint == "POST /enrollments").sum()),
    "per_endpoint_ms": {e: {**r1(pct(g.ms)), "n": int(len(g))} for e, g in req.groupby("endpoint")},
}
summ = json.loads((RESULTS / "summary.json").read_text())
stats["idempotent_replays"] = int(summ["metrics"].get("idempotent_replays", {}).get("count", 0))
res = pd.DataFrame(samples)
if len(res):
    stats["resources"] = {
        "java_cpu_peak_pct": round(float(res.java_cpu.max()), 1), "java_cpu_mean_pct": round(float(res.java_cpu.mean()), 1),
        "java_rss_peak_mb": round(float(res.java_rss_mb.max()), 1), "pg_rss_peak_mb": round(float(res.pg_rss_mb.max()), 1),
        "system_cpu_peak_pct": round(float(res.sys_cpu.max()), 1), "system_cpu_mean_pct": round(float(res.sys_cpu.mean()), 1),
        "pg_connections_peak": int(res.pg_conns.max()) if res.pg_conns.notna().any() else None}
vm = psutil.virtual_memory()
stats["environment"] = {"os": platform.platform(), "cpu_count": os.cpu_count(), "ram_gb": round(vm.total / 2**30, 1),
    "java": sh("java -version 2>&1 | head -1", show=False).stdout.strip(),
    "k6": sh("k6 version", show=False).stdout.strip(), "postgres": sh("psql --version", show=False).stdout.strip(),
    "git_hash": GIT_HASH, "vus": VUS, "students": STUDENTS, "k6_exit_code": k6.returncode,
    "started_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(t_start)),
    "ended_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(t_end))}
(RESULTS / "stats.json").write_text(json.dumps(stats, indent=2, ensure_ascii=False))
print(json.dumps({k: v for k, v in stats.items() if k != "per_endpoint_ms"}, indent=2, ensure_ascii=False))

## Bước 10. Biểu đồ

| Hình | Nội dung |
|---|---|
| 01 | Thông lượng và số VU theo thời gian |
| 02 | Độ trễ p50/p95/p99 theo cửa sổ 5 giây |
| 03 | Độ trễ theo endpoint |
| 04 | Phân bố độ trễ: đăng nhập so với endpoint khác |
| 05 | Mã trạng thái theo endpoint |
| 06 | Tỷ lệ lỗi theo thời gian |
| 07 | CPU và RAM |
| 08 | Phân phối tích lũy độ trễ theo endpoint |
| 09 | Bản đồ nhiệt p95 theo endpoint và thời gian |
| 10 | Thông lượng xếp chồng theo endpoint |
| 11 | Số kết nối PostgreSQL theo thời gian |
| 12 | Bảng điểm QC: đo được so với ngưỡng (ở Bước 12) |

Ảnh lưu trong `charts/` của gói kết quả.

In [ ]:
# 10. Biểu đồ
import matplotlib
if "google.colab" not in sys.modules and "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
C = {"p50": "#2a9d8f", "p95": "#e9a23b", "p99": "#d1495b", "vu": "#4f6d9a", "ok": "#2a9d8f", "err": "#d1495b"}
def save(name):
    plt.tight_layout(); plt.savefig(CHARTS / name, dpi=130); plt.show(); plt.close()

# 10.1 Thông lượng và số VU
fig, ax = plt.subplots(figsize=(10, 4))
ps = per_sec.reindex(range(int(req.sec.max()) + 1), fill_value=0)
ax.plot(ps.index, ps.values, color=C["ok"], lw=1.2, label="Yêu cầu/giây")
ax.set_xlabel("Giây kể từ khi bắt đầu"); ax.set_ylabel("req/s")
vu = raw[raw.metric_name == "vus"].groupby(raw.sec.astype(int)).metric_value.max()
ax2 = ax.twinx(); ax2.plot(vu.index, vu.values, color=C["vu"], lw=1.2, ls="--", label="VU"); ax2.set_ylabel("VU"); ax2.grid(False)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax.legend(h1 + h2, l1 + l2, loc="upper left")
ax.set_title("Thông lượng và số người dùng ảo theo thời gian"); save("01-thong-luong-vu.png")

# 10.2 Độ trễ theo thời gian
w = req.groupby((req.sec // 5) * 5).ms
lat = pd.DataFrame({"p50": w.quantile(.5), "p95": w.quantile(.95), "p99": w.quantile(.99)})
fig, ax = plt.subplots(figsize=(10, 4))
for k in ("p50", "p95", "p99"): ax.plot(lat.index, lat[k], color=C[k], lw=1.5, label=k)
ax.set_yscale("log"); ax.set_xlabel("Giây"); ax.set_ylabel("ms (thang log)"); ax.legend()
ax.set_title("Độ trễ p50, p95, p99 theo cửa sổ 5 giây"); save("02-do-tre-theo-thoi-gian.png")

# 10.3 Độ trễ theo endpoint
pe = pd.DataFrame(stats["per_endpoint_ms"]).T.sort_values("p95")
fig, ax = plt.subplots(figsize=(10, 4.5)); x = np.arange(len(pe)); bw = .27
for i, k in enumerate(("p50", "p95", "p99")): ax.bar(x + (i - 1) * bw, pe[k], bw, color=C[k], label=k)
ax.set_xticks(x); ax.set_xticklabels(pe.index, rotation=25, ha="right"); ax.set_yscale("log"); ax.set_ylabel("ms (thang log)"); ax.legend()
ax.set_title("Độ trễ theo endpoint"); save("03-do-tre-theo-endpoint.png")

# 10.4 Phân bố độ trễ: đăng nhập so với các endpoint khác
fig, ax = plt.subplots(figsize=(10, 4))
bins = np.logspace(np.log10(max(req.ms.min(), 1)), np.log10(req.ms.max()), 60)
ax.hist(req[req.endpoint != "POST /auth/login"].ms, bins=bins, alpha=.7, color=C["ok"], label="Các endpoint khác")
ax.hist(login.ms, bins=bins, alpha=.7, color=C["p99"], label="POST /auth/login")
ax.set_xscale("log"); ax.set_xlabel("ms (thang log)"); ax.set_ylabel("Số yêu cầu"); ax.legend()
ax.set_title("Phân bố độ trễ"); save("04-phan-bo-do-tre.png")

# 10.5 Mã trạng thái theo endpoint
sc = req.groupby(["endpoint", "status"]).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(10, 4.5)); bottom = np.zeros(len(sc)); cmap = plt.get_cmap("tab10")
for i, s in enumerate(sc.columns):
    ax.bar(sc.index, sc[s], bottom=bottom, label=str(s), color=cmap(i % 10)); bottom += sc[s].values
ax.set_ylabel("Số yêu cầu")
plt.setp(ax.get_xticklabels(), rotation=25, ha="right"); ax.legend(title="HTTP", ncol=2)
ax.set_title("Mã trạng thái theo endpoint"); save("05-ma-trang-thai.png")

# 10.6 Tỷ lệ lỗi theo thời gian
bad = (~req.status.between(200, 299) & ~req.status.isin([409, 422])).astype(int)
er = bad.groupby((req.sec // 5) * 5).mean() * 100
fig, ax = plt.subplots(figsize=(10, 3.6)); ax.plot(er.index, er.values, color=C["err"], lw=1.5)
ax.set_xlabel("Giây"); ax.set_ylabel("% lỗi (không tính 409, 422)"); ax.set_title("Tỷ lệ lỗi theo thời gian"); save("06-ty-le-loi.png")

# 10.7 Tài nguyên
if len(res):
    rt = res.t - res.t.iloc[0]
    fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
    axs[0].plot(rt, res.sys_cpu, color=C["vu"], label="Toàn máy"); axs[0].plot(rt, res.java_cpu, color=C["p95"], label="Java (có thể > 100%)")
    axs[0].set_ylabel("CPU %"); axs[0].set_xlabel("Giây"); axs[0].legend(fontsize=8); axs[0].set_title("CPU")
    axs[1].plot(rt, res.java_rss_mb, color=C["p95"], label="Java RSS"); axs[1].plot(rt, res.pg_rss_mb, color=C["ok"], label="PostgreSQL RSS")
    axs[1].set_ylabel("MB"); axs[1].set_xlabel("Giây"); axs[1].legend(fontsize=8); axs[1].set_title("RAM")
    save("07-tai-nguyen.png")
# 10.8 Phân phối tích lũy (ECDF) độ trễ theo endpoint
fig, ax = plt.subplots(figsize=(10, 4.5))
for e, g in req.groupby("endpoint"):
    v = np.sort(g.ms.values); ax.plot(v, np.arange(1, len(v) + 1) / len(v) * 100, lw=1.6, label=e)
ax.axvline(3000, color=C["p99"], ls=":", lw=1.2); ax.text(3000, 4, " ngưỡng p95 = 3000 ms", color=C["p99"], fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("ms (thang log)"); ax.set_ylabel("% yêu cầu hoàn tất"); ax.legend(fontsize=8, loc="lower right")
ax.set_title("Phân phối tích lũy độ trễ theo endpoint"); save("08-ecdf-do-tre.png")

# 10.9 Bản đồ nhiệt p95 theo endpoint và thời gian
hm = req.assign(win=(req.sec // 10 * 10).astype(int)).pivot_table(
    index="endpoint", columns="win", values="ms", aggfunc=lambda s: s.quantile(.95))
fig, ax = plt.subplots(figsize=(10, 3.8))
im = ax.imshow(hm.values, aspect="auto", cmap="YlOrRd",
               norm=matplotlib.colors.LogNorm(vmin=max(np.nanmin(hm.values), 1), vmax=np.nanmax(hm.values)))
ax.set_yticks(range(len(hm.index))); ax.set_yticklabels(hm.index)
ax.set_xticks(range(len(hm.columns))); ax.set_xticklabels(hm.columns)
ax.set_xlabel("Giây (cửa sổ 10 giây)"); ax.grid(False); fig.colorbar(im, ax=ax, label="p95 (ms)")
ax.set_title("Bản đồ nhiệt p95 theo endpoint và thời gian"); save("09-nhiet-p95.png")

# 10.10 Thông lượng xếp chồng theo endpoint
tp = (req.assign(s=req.sec.astype(int)).groupby(["s", "endpoint"]).size().unstack(fill_value=0)
         .reindex(range(int(req.sec.max()) + 1), fill_value=0))
fig, ax = plt.subplots(figsize=(10, 4))
ax.stackplot(tp.index, tp.T.values, labels=tp.columns, alpha=.85)
ax.set_xlabel("Giây"); ax.set_ylabel("req/s"); ax.legend(fontsize=8, loc="upper left", ncol=2)
ax.set_title("Thông lượng theo endpoint"); save("10-thong-luong-endpoint.png")

# 10.11 Số kết nối PostgreSQL
if len(res) and res.pg_conns.notna().any():
    pc = res.dropna(subset=["pg_conns"])
    fig, ax = plt.subplots(figsize=(10, 3.6))
    ax.step(pc.t - res.t.iloc[0], pc.pg_conns, where="post", color=C["vu"], lw=1.6)
    ax.set_xlabel("Giây"); ax.set_ylabel("Kết nối"); ax.set_title("Số kết nối PostgreSQL theo thời gian"); save("11-ket-noi-db.png")
print(sorted(p.name for p in CHARTS.iterdir()))

## Bước 11. Kiểm bất biến dữ liệu

Dừng backend rồi chạy `docker/verify/invariants.sql`. Mỗi truy vấn phải trả **0 dòng**:

| Mã | Bất biến |
|---|---|
| a | `enrolled_count` khớp số dòng `ENROLLED` |
| b | `enrolled_count` trong `[0, capacity]` (không vượt sức chứa) |
| c | Không đăng ký hai lớp cùng môn trong một kỳ |
| d | Không trùng ca học |
| e | Không vượt hạn mức tín chỉ |
| f | Không bản ghi idempotency `PROCESSING` quá hạn |
| g | Chỉ một đợt `OPEN` |

In [ ]:
# 11. Kiểm bất biến dữ liệu sau tải (invariants.sql), dừng backend trước
import re
app.terminate(); app.wait(timeout=30)
inv = sh("psql -U dkhp_app -d dkhp -f docker/verify/invariants.sql", env=PG_ENV, cwd=SRC, show=False, check=False)
(RESULTS / "invariants_output.txt").write_text(inv.stdout + inv.stderr)
blocks = re.split(r"(?m)^\[([a-g])\] ", inv.stdout)[1:]
verdict = {blocks[i]: int(re.search(r"\((\d+) rows?\)", blocks[i + 1]).group(1)) for i in range(0, len(blocks), 2)}
extra = {}
for key, q in {"enrollment_total": "SELECT count(*) FROM enrollment",
               "enrollment_enrolled": "SELECT count(*) FROM enrollment WHERE status='ENROLLED'",
               "idempotency_total": "SELECT count(*) FROM idempotency_record",
               "idempotency_processing": "SELECT count(*) FROM idempotency_record WHERE state='PROCESSING'",
               "students_with_enrollment": "SELECT count(DISTINCT student_id) FROM enrollment WHERE status='ENROLLED'"}.items():
    extra[key] = int(sh(f'psql -U dkhp_app -d dkhp -Atc "{q}"', env=PG_ENV, show=False).stdout.strip())
final = {"violating_rows_per_invariant": verdict, "result": "PASS" if verdict and all(v == 0 for v in verdict.values()) else "FAIL", **extra}
(RESULTS / "invariants.json").write_text(json.dumps(final, indent=2))
print(inv.stdout); print(json.dumps(final, indent=2))

## Bước 12. Chấm QC

So kết quả đo với ngưỡng trong `options.thresholds` của `full_scenario.js` và với các bất biến dữ liệu.

| Chỉ số | Ngưỡng |
|---|---|
| p95 latency | < 3000 ms |
| p99 latency | < 5000 ms |
| Tỷ lệ lỗi (không tính 409, 422) | < 10 % |
| Bất biến a..g | PASS |
| idempotency `PROCESSING` mồ côi | 0 |

Ngưỡng chính thức (D6) chưa được chốt, nên đây chỉ là mốc tham chiếu. Ghi `qc.json`.

In [ ]:
# 12. Chấm QC: ngưỡng lấy từ options.thresholds của load/k6/full_scenario.js + bất biến dữ liệu
QC_P95_MS, QC_P99_MS, QC_ERR_PCT = 3000, 5000, 10.0
checks = [
    ("p95 latency toàn hệ thống", stats["latency_ms"]["p95"], f"< {QC_P95_MS} ms", stats["latency_ms"]["p95"] < QC_P95_MS),
    ("p99 latency toàn hệ thống", stats["latency_ms"]["p99"], f"< {QC_P99_MS} ms", stats["latency_ms"]["p99"] < QC_P99_MS),
    ("Tỷ lệ lỗi (không tính 409, 422)", stats["error_rate_excl_business_pct"], f"< {QC_ERR_PCT} %", stats["error_rate_excl_business_pct"] < QC_ERR_PCT),
    ("Bất biến dữ liệu a..g", final["result"], "PASS", final["result"] == "PASS"),
    ("Không bản ghi idempotency PROCESSING mồ côi", final["idempotency_processing"], "= 0", final["idempotency_processing"] == 0),
]
qc_rows = [{"chi_so": n, "ket_qua": v, "nguong": t, "dat": bool(ok)} for n, v, t, ok in checks]
qc_out = {"checks": qc_rows, "overall": "PASS" if all(r["dat"] for r in qc_rows) else "FAIL",
          "note": "Ngưỡng p95/p99/lỗi lấy từ full_scenario.js; ngưỡng chính thức D6 chưa chốt."}
(RESULTS / "qc.json").write_text(json.dumps(qc_out, indent=2, ensure_ascii=False))
print(pd.DataFrame(qc_rows).to_string(index=False)); print("QC tổng:", qc_out["overall"])

fig, axs = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, (name, val, lim, unit) in zip(axs, [("p95", stats["latency_ms"]["p95"], QC_P95_MS, "ms"),
                                            ("p99", stats["latency_ms"]["p99"], QC_P99_MS, "ms"),
                                            ("Tỷ lệ lỗi", stats["error_rate_excl_business_pct"], QC_ERR_PCT, "%")]):
    ok = val < lim
    ax.bar(["Đo được", "Ngưỡng"], [val, lim], color=[C["ok"] if ok else C["err"], "#9aa5b1"])
    for x, v in zip((0, 1), (val, lim)): ax.text(x, v, f"{v:,.1f}", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel(unit); ax.set_title(f"{name}: {'ĐẠT' if ok else 'KHÔNG ĐẠT'}")
fig.suptitle(f"Chấm QC: {qc_out['overall']} | bất biến dữ liệu: {final['result']}", fontweight="bold")
save("12-cham-qc.png")


## Bước 13. Đóng gói kết quả

Gom `stats.json`, `qc.json`, `invariants.json`, `resources.csv`, `requests.csv.gz`, log, biểu đồ và kịch bản k6 vào `/kaggle/working/load_results.zip`. Tải về ở tab **Output**, giải nén vào `load/results/` của repo.

In [ ]:
# 13. Đóng gói kết quả vào /kaggle/working (tab Output)
shutil.copy(SRC / "load" / "k6" / "full_scenario.js", RESULTS / "full_scenario.js")
zip_path = shutil.make_archive(str(OUT_DIR / "load_results"), "zip", RESULTS)
print("Đã đóng gói:", zip_path, round(os.path.getsize(zip_path) / 1e6, 1), "MB")
